<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP10.2B/MP10.2B_day_of_operation.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP10.2B · A Day of Operation Under an Hourly Price

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Physics-Informed Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/")
PROJECT = (['mp102b.py', 'MP10.2B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP10.2B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex10.2-solid-oxide-cell", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex10.2-solid-oxide-cell", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp102b']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp102b as mp
torch.set_default_dtype(torch.float64)                   # a second derivative of a network needs double precision
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP10.2B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex10.2's problem.py, by its name in Ex10.2
import os, shutil, urllib.request
_w = "Ex10.2_weights.pt"                                 # Ex10.2's trained network, the model of an hour
_local = os.path.join("..", "..", "exercises", "Ex10.2-solid-oxide-cell", _w)
if "google.colab" in sys.modules or not os.path.exists(_w):
    if "google.colab" not in sys.modules and os.path.exists(_local):
        shutil.copy(_local, _w)
    else:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/"
                                   "exercises/Ex10.2-solid-oxide-cell/" + _w, _w)
    print("fetched", _w)
G = np.load("MP10.2B_truth.npz")                         # the ground truth (part C explains it)

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

An electrolyser earns its keep by following the electricity market. The price changes every hour - low at night and when the sun or the wind is strong, high in the morning and the evening peaks - and the cell can change its operating point within minutes. A plant that sells its hydrogen under a **contract** for a fixed amount a day can choose **when** to make it: more in the cheap hours, less in the dear ones. But a solid oxide cell cannot be pushed freely. Above the thermoneutral voltage it heats along its length, below it cools, and the ceramic layers tolerate only a limited temperature change; and the harder it is run, the faster it ages.

So the question is a **schedule**: for each of the 24 hours of tomorrow, the cell voltage and the inlet temperature that make the contracted hydrogen at the least cost of electricity and ageing, without exceeding the temperature limit in any hour. A new price curve arrives every day (and an intraday market changes it during the day), so the schedule is computed again and again. That is where a fast model pays.

This mini project computes the schedule for any hourly price curve, through the network of Ex10.2.

## A2 · The cell, and what is new against Ex10.2

The cell is Ex10.2's, unchanged: 100 cm² of electrode, the fuel channel with its three coupled laws (the steam balance, the energy balance with the heat $i\,(V - V_{tn})$, and the charge transfer at every point), the cost of an hour of operation (electricity, less the hydrogen sold, plus the ageing of the stack), and the 10 K limit on the temperature change along the cell.

**What is new.** Ex10.2's part B chose **one** number, the cell voltage, for **one** hour at a fixed inlet temperature of 800 °C and a fixed electricity price. Here:

* the electricity price changes **every hour**, by a day-ahead curve;
* the operator chooses **the cell voltage and the inlet temperature for each hour**: 48 numbers;
* the hydrogen is sold under a **contract** for the day: what the cell makes in 24 hours at 1.20 V with the gas entering at 800 °C. The hydrogen's revenue is then fixed, and the choice is about **when** to make it;
* in every hour the temperature change along the cell stays within 10 K.

Four price curves are given, all **invented for this project** with the shapes of a day-ahead market: an autumn weekday, a sunny spring day (free electricity at noon), a windy night, and a winter cold snap (prices two to three times higher). None of them is used to build any model: each is a **new case** for every method.

In [ ]:
# The four hourly price curves and the data of the day
fig, ax = plt.subplots(figsize=(9, 3.4))
for name, price in mp.PRICES.items():
    ax.step(np.arange(25), np.r_[price, price[-1]] * 100, where="post", label=name)
ax.set_xlabel("hour of the day"); ax.set_ylabel("electricity price [euro cent/kWh]"); ax.set_xticks(range(0, 25, 3)); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
for name, price in mp.PRICES.items():
    print(f"{name:>18}: {100*price.min():5.2f} to {100*price.max():5.2f} cent/kWh, mean {100*price.mean():5.2f}")
rows = [("cell", f"Ex10.2's, {pb.A_CELL:.0f} cm², every kinetic, transport and ageing value Ex10.2's estimate"),
        ("decisions", f"V = {pb.V_LOW:.2f} to {pb.V_HIGH:.2f} V and T_in = {pb.T_LOW-273.15:.0f} to {pb.T_HIGH-273.15:.0f} °C for each of 24 hours"),
        ("contract", f"{1e3*float(G['contract_kg']):.2f} g of hydrogen a day: 24 h at {mp.V_CONTRACT:.2f} V and {mp.T_CONTRACT-273.15:.0f} °C"),
        ("limit", f"largest temperature change along the cell {mp.LIMIT:.0f} K in every hour (Ex10.2's estimate)"),
        ("ageing", f"stack {pb.STACK_COST:.0f} euro/kW, end of life at +{pb.DASR_EOL:.2f} ohm cm² (Ex10.2's estimates)")]
for k, v in rows:
    print(f"{k:>18}  {v}")

**What you should see.** Four staircases: the autumn weekday from 2 to 14 cent/kWh with its evening peak, the sunny spring day falling to zero at noon, the windy night cheapest from 22:00 to 05:00, and the winter cold snap from 9.5 to 33 cent/kWh. And the data: the contract is 93.33 g of hydrogen a day, about 3.9 g an hour.

## A3 · The model of an hour

Each hour is judged at its **steady state**, by Ex10.2's coupled model of the fuel channel at the hour's cell voltage $V_t$ and inlet temperature $T_t$:

| law | what it says | what it gives the day |
|---|---|---|
| steam balance | steam is used where the current flows | the steam the cell can split |
| energy balance | the cell heats above the thermoneutral voltage 1.29 V and cools below it | the largest temperature change $\Delta T_t$ |
| charge transfer | at every point the applied voltage covers the Nernst potential and the losses | the current density, hence the current $I_t$ |

From the fields of an hour come four numbers: the **electricity** $E_t = V_t I_t$ × 1 h [kWh], the **hydrogen** $H_t = I_t M_{H_2}/2F$ × 1 h [kg] (Faraday), the **ageing** $A_t$ [cent], the share of the stack that ages in the hour (Ex10.2's law, which grows with the current density and the temperature), and $\Delta T_t$. The day is then

$$\min_{V_1..V_{24},\ T_1..T_{24}} \ C = \sum_{t=1}^{24} \big( 100\,p_t\,E_t + A_t \big) \quad \text{such that} \quad \sum_t H_t = H_\mathrm{contract}, \qquad \Delta T_t \le 10\ \mathrm{K}\ \text{for every}\ t$$

with $p_t$ the hour's price in euro/kWh and $C$ in euro cents. The hydrogen sold does not appear: under the contract it earns the same whatever the schedule. Each hour is quasi-steady, as in Ex10.2: moving between operating points takes a few minutes of thermal transient, which is not modelled; and the ageing prices the hour without feeding the grown resistance back into the model.

The cell below maps the four numbers over the operating range, from the ground truth's table (part C).

In [ ]:
# The four numbers of one hour over the operating range, from the ground truth's table (finite volumes on 800 cells)
VV, TT = np.meshgrid(G["VB"], G["TB"] - 273.15, indexing="ij")
fig, ax = plt.subplots(1, 4, figsize=(17, 3.6))
for a, (key, label) in zip(ax, (("hydrogen", "hydrogen [g/h]"), ("electricity", "electricity [kWh/h]"),
                                 ("ageing", "ageing [cent/h]"), ("dT", "largest temperature change [K]"))):
    f = G[f"table_{key}"] * (1e3 if key == "hydrogen" else 1)
    cs = a.contourf(VV, TT, f, levels=20, cmap="viridis"); plt.colorbar(cs, ax=a)
    a.contour(VV, TT, G["table_dT"], levels=[mp.LIMIT], colors="w", linewidths=1.5)
    a.set_title(label, fontsize=9); a.set_xlabel("cell voltage [V]")
ax[0].set_ylabel("inlet temperature [°C]")
plt.tight_layout(); plt.show()
ok = G["table_dT"] <= mp.LIMIT
print(f"one hour makes {1e3*G['table_hydrogen'].min():.2f} to {1e3*G['table_hydrogen'].max():.2f} g of hydrogen; "
      f"within the 10 K limit at most {1e3*G['table_hydrogen'][ok].max():.2f} g")
for T in (750.0, 800.0, 850.0):
    b = int(np.argmin(np.abs(G["TB"] - 273.15 - T))); Vok = G["VB"][G["table_dT"][:, b] <= mp.LIMIT]
    print(f"inlet at {T:.0f} °C: the 10 K limit allows {Vok.min():.3f} to {Vok.max():.3f} V")

**What you should see.** The hydrogen rises with the voltage and with the temperature (a hotter electrolyte conducts better); so does the electricity. The ageing rises steeply with both. The white line is the 10 K limit: it closes a **band of voltages around the thermoneutral 1.29 V** - below the band the cell cools by more than 10 K, above it heats by more - and the band is narrower the hotter the gas: from about 1.24 to 1.32 V at 750 °C. One hour makes 2.42 to 6.23 g over the whole range, at most 6.04 g within the limit (at 850 °C), 4.46 g at 750 °C; the contract asks for 3.89 g an hour on average. The schedule therefore moves inside this band: high in the cheap hours, low in the dear ones.

## A4 · Why there is no exact solution

Every number of an hour comes from Ex10.2's three coupled laws, which have no closed form; the day is a constrained optimisation over 48 of these. No formula gives the schedule.

What is exact is its **structure**. Price the hydrogen with a **Lagrange multiplier** $\lambda$ [cent/kg]: then

$$\min_{V_t,\,T_t}\ \big(100\,p_t E_t + A_t - \lambda\,H_t\big) \quad \text{with}\ \Delta T_t \le 10\ \mathrm{K}$$

is a separate problem for every hour - the hours do not interact - and the contract fixes $\lambda$: the hydrogen of the day rises with $\lambda$, so one number is found by bisection. $\lambda$ is the **marginal cost of the contract's hydrogen**: what one more kilogram a day would cost in electricity and ageing. The ground truth (part C) uses this structure; any method may.

**A check that follows from it.** Under a flat price every hour is the same problem, so the optimal schedule is one operating point held all day. Part C confirms that the ground truth's machinery returns exactly that.

## A5 · What is asked

| quantity | for | required |
|---|---|---|
| the day's cost $C$ of your schedule, judged by finite volumes on 800 cells | each of the four price curves | within **0.5 %** of the ground truth's optimum |
| the hydrogen of the day, judged the same way | the same | within **0.5 %** of the contract |
| the largest temperature change in any hour, judged the same way | the same | at most **10.2 K** (the 10 K limit and an agreed tolerance of 0.2 K) |
| the time to answer a new price curve online | one Colab T4 runtime | at least **10 times shorter** than the reference model's (part C measures it) |

The schedule is computed **through a network**: the parametric network of Ex10.2, which returns the fields of the channel for any cell voltage and inlet temperature, is the model of every hour, and the 48 numbers are found by a gradient-based optimiser through it, its gradients taken by autograd. The network is trained once, offline (Ex10.2 did it); a new price curve costs one optimisation through it. Your schedule is then **judged by the finite volumes**, not by the network: an error of the network becomes an error of the schedule.

**Why 0.5 %.** Holding the best single operating point all day costs 3.4 to 5.9 % more than the optimal schedule on these curves (part C): an answer within 0.5 % has found most of what the hourly schedule is worth. The reference model, SLSQP on Ex10.2's engineering grid, meets it (part C).

---

# Part B · The Data and the Physics

## B1 · The network: Ex10.2's model of an hour

The network of Ex10.2's part A takes three inputs, each scaled to [0, 1], and returns the three fields of the channel:

$$(x,\ v,\ \tau)\ \longmapsto\ \big(y,\ \theta,\ i\big)$$

$x$ the position along the channel, $v = (V - 1.15)/0.20$ the cell voltage, $\tau = (T_\mathrm{in} - 750\ °\mathrm{C})/100$ K the inlet temperature; $y$ the steam fraction, $\theta = (T - T_\mathrm{in})/50$ K the temperature change, $i$ the current density in A/cm². Four layers of 64 tanh neurons; the inlet values built in ($y = 0.90 + x\,\mathcal{N}_1$, $\theta = x\,\mathcal{N}_2$) and the current density kept between zero and the limiting current ($i = i_L(y)\,\sigma(\mathcal{N}_3)$). It was trained by Ex10.2 with the three laws as its loss - 9000 Adam steps and L-BFGS, **2005 s on a 2-core CPU** - and its weights are `Ex10.2_weights.pt`, fetched beside the libraries.

The **operating point is an input**: the same network answers every hour of every day, and its derivatives with respect to $v$ and $\tau$ come from autograd. That is what makes a gradient through the day possible.

From the fields along the channel the four numbers of an hour are integrals and a maximum: the current $I = A\int_0^1 i\,dx$, the ageing $\int_0^1 k\,e^{-E_d/RT}\,i^{1.5}\,dx$ (Ex10.2's law), and $\Delta T = 50\ \mathrm{K}\,\max_x|\theta|$. `hour_numbers` in part D takes them by Simpson's rule on 51 points. The cell below loads the network and compares the four numbers with the ground truth's finite volumes on 800 cells, at 231 operating points spread over the range.

In [ ]:
# Ex10.2's network: load it, and compare its four numbers of an hour with the ground truth's table
from pinn_core import MLP

class Cell(torch.nn.Module):                             # Ex10.2's network, as its notebook defines it
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=3, n_out=3, n_hidden=64, n_layers=4)   # x, v, tau in; three outputs
    def forward(self, p):
        o = self.net(2 * p - 1)                          # inputs centred on [-1, 1] for tanh
        x = p[:, 0:1]
        y = pb.Y_IN + x * o[:, 0:1]                      # steam fraction: the inlet value, exactly
        th = x * o[:, 1:2]                               # scaled temperature: zero at the inlet, exactly
        i = pb.i_lim(y) * torch.sigmoid(o[:, 2:3])       # current density between zero and its limit, A/cm2
        return y, th, i

NX = 51                                                  # points along the channel for the integrals
XQ = torch.linspace(0, 1, NX, device=DEVICE).reshape(-1, 1)
WS = torch.ones(NX, device=DEVICE); WS[1:-1:2] = 4; WS[2:-1:2] = 2; WS = WS / (3 * (NX - 1))   # Simpson's weights

def hour_numbers(model, V, T_in):
    # the four numbers of each hour, through the network: V [V] and T_in [K] are tensors of n hours
    n = V.shape[0]
    v = ((V - pb.V_LOW) / (pb.V_HIGH - pb.V_LOW)).repeat_interleave(NX).reshape(-1, 1)
    tau = ((T_in - pb.T_LOW) / (pb.T_HIGH - pb.T_LOW)).repeat_interleave(NX).reshape(-1, 1)
    y, th, i = model(torch.cat([XQ.repeat(n, 1), v, tau], 1))           # n hours x NX points, one batch
    i, th = i.reshape(n, NX), th.reshape(n, NX)
    T = T_in.reshape(-1, 1) + pb.DT_SCALE * th                          # temperature along the channel [K]
    current = (i * WS).sum(1) * pb.A_CELL                               # A
    ageing = (pb.K_DEG * torch.exp(-pb.E_DEG / (pb.R_GAS * T)) * i.clamp(min=0) ** pb.N_DEG * WS).sum(1)
    return dict(electricity=V * current / 1000,                         # kWh in the hour
                ageing=100 * pb.STACK_COST * pb.P_RATED * ageing / pb.DASR_EOL,   # cent in the hour
                hydrogen=current / (2 * pb.FARADAY) * pb.M_H2 * 3600,   # kg in the hour (Faraday)
                dT=(pb.DT_SCALE * th).abs().max(1).values)              # K, the largest change along the cell

ex102 = Cell().to(DEVICE)
ex102.load_state_dict(torch.load("Ex10.2_weights.pt", map_location=DEVICE))   # trained by Ex10.2, offline
VV, TT = np.meshgrid(G["VB"][::4], G["TB"][::4], indexing="ij")             # 21 x 11 operating points
with torch.no_grad():
    h = hour_numbers(ex102, torch.tensor(VV.ravel(), device=DEVICE), torch.tensor(TT.ravel(), device=DEVICE))
for key, unit in (("hydrogen", "relative"), ("electricity", "relative"), ("ageing", "relative"), ("dT", "K")):
    truth = G[f"table_{key}"][::4, ::4].ravel(); net_ = h[key].cpu().numpy()
    e = np.abs(net_ / truth - 1) if unit == "relative" else np.abs(net_ - truth)
    print(f"{key:>12}: worst error {100*e.max():.2f} %, mean {100*e.mean():.3f} %" if unit == "relative"
          else f"{key:>12}: worst error {e.max():.2f} K, mean {e.mean():.3f} K")

**What you should see.** Over 231 operating points the network's hydrogen (and current) is within 0.16 % of the finite volumes on 800 cells (0.024 % on average), its electricity likewise, its ageing within 0.22 %, its temperature change within 0.20 K (0.04 K on average); the largest errors are at the edges of the range. These are the errors your schedule inherits: a schedule that meets the contract exactly **in the network** makes slightly more or less hydrogen in the finite volumes, and sits slightly off the limit. Both are inside the requirements, with room to spare; part D can tighten them (a margin on the limit, a correction of the contract), or train a network of its own.

## B2 · The optimisation the network is used for

The objective and the constraints of section A3 become, through the network, **differentiable functions of the 48 numbers**: autograd gives $\partial C/\partial V_t$, $\partial H_t/\partial T_t$ and so on, at the cost of one backward pass through 24 × 51 points. Three things must be decided, and each has a standard answer:

| what | how it can be done | what to watch |
|---|---|---|
| the bounds, 1.15 to 1.35 V and 750 to 850 °C | optimise unbounded numbers $z$ and map them, $V = 1.15 + 0.20\,\sigma(z)$; or clip after each step | the network was trained only inside the range |
| the contract $\sum_t H_t = H_\mathrm{contract}$ | a multiplier $\lambda$ found by bisection (section A4); or an **augmented Lagrangian**, $C + \lambda g + \tfrac{\rho}{2} g^2$ with $g$ the shortfall and $\lambda \leftarrow \lambda + \rho g$ after each solve | at the optimum some hours are **indifferent** to their operating point, so for a fixed $\lambda$ the hydrogen of the day is not pinned: a quadratic term pins it |
| the limit $\Delta T_t \le 10$ K | a penalty $\mu \sum_t \max(0, \Delta T_t - 10 + m)^2$ with a small margin $m$; or a multiplier per hour | the network's $\Delta T$ is 0.1 K off at worst: a margin keeps the finite volumes under 10.2 K |

The optimiser can be Adam (many cheap steps), L-BFGS (few, with curvature) on a penalised or augmented objective, or a constrained method such as SLSQP - the reference model's own - fed with the network's exact gradients. Which is fastest for the accuracy asked is part of your work: the methods differ by a factor of fifty in the number of evaluations of the network.

## B3 · The price curves

The four curves of section A2 are the **test cases**: they are given here so that you can check your optimiser, but nothing is trained on them, and your optimiser must work for any curve, as it would for tomorrow's. The cell below shows how the four differ in what matters: the spread between the cheap and the dear hours.

In [ ]:
# What the four curves offer: the spread of prices, and how many hours are cheap
for name, price in mp.PRICES.items():
    cheap = int(np.sum(price < np.median(price)))
    print(f"{name:>18}: mean {100*price.mean():5.2f} cent/kWh, spread {100*np.ptp(price):5.2f}, "
          f"cheapest hour {int(np.argmin(price)):2d}:00, dearest {int(np.argmax(price)):2d}:00")

**What you should see.** Spreads from 12 cent/kWh (autumn) to 24 (winter), the cheapest hour at noon on the spring day and in the night on the windy day, the dearest at 17:00 or 18:00 on all four. The larger the spread, the more the schedule can save against a constant operating point.

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two solvers, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | what an engineer would run for a new price curve: SLSQP on the 48 numbers, with the coupled finite volumes as the model of every hour | the optimum found with a model a hundred times more exact, by the structure of section A4 |
| model of an hour | Ex10.2's finite volumes on **50 cells** (its agreed accuracy: 0.01 in steam fraction, 0.5 K) | finite volumes on **800 cells**, tabulated on 81 × 41 operating points |
| its job | to be **timed** against the network: the online cost of a new price curve | to **score** the network's schedule and the reference's |

## C2 · The ground truth, step by step

1. **The table.** Every hour's four numbers at 81 voltages × 41 inlet temperatures (2.5 mV × 2.5 K), by the finite volumes on 800 cells: 3321 solves, about 2 minutes on a CPU.
2. **Smooth.** Bicubic splines through the table give the four numbers, and their derivatives, at any operating point.
3. **One hour, given $\lambda$.** On the splines, SLSQP minimises $100\,p_t E_t + A_t - \lambda H_t$ under $\Delta T_t \le 10$ K, started from the best of 41 × 21 grid points.
4. **The day.** The hydrogen of the day rises with $\lambda$; bisection (50 halvings) finds the $\lambda$ that makes the contract.
5. **Checked again.** The schedule found is solved once more, hour by hour, by the finite volumes on 800 cells: its cost, its hydrogen and its temperature changes are those of the solver, not of the splines.

The convergence study repeats steps 1 to 4 with tables on 100, 200, 400 and 800 cells, and with the 800-cell table on coarser operating grids (10 mV × 10 K and 5 mV × 5 K).

In [ ]:
# The ground truth: its convergence, its checks, and the optimal day for each price curve
print("day's cost on the autumn weekday [cent], by the model of an hour:")
for n, c in zip(G["conv_cells"], G["conv_cells_cost"]):
    print(f"   finite volumes on {int(n):3d} cells: {c:.4f}")
d = np.diff(G["conv_cells_cost"]) / G["conv_cells_cost"][-1]
print("   change between cell counts, relative:", " ".join(f"{v:+.1e}" for v in d))
for s, c in zip(G["conv_grid_steps"], G["conv_grid_cost"]):
    print(f"   operating grid every {1e3*s:.1f} mV and {1e3*s:.1f} K: {c:.6f}")
print(f"finite volumes' conservation on 800 cells: steam {abs(float(G['balance_steam'])):.1e}, heat {abs(float(G['balance_heat'])):.1e} (scaled)")
print(f"under a flat price the optimal schedule spreads over {1e3*float(G['flat_spread_V']):.2f} mV and {float(G['flat_spread_T']):.2f} K: one operating point")
print()
for name in mp.PRICE_NAMES:
    k = name.replace(" ", "_")
    print(f"{name:>18}: day's cost {float(G[k+'_cost']):.2f} cent, hydrogen {1e3*float(G[k+'_hydrogen']):.2f} g "
          f"(contract {1e3*float(G['contract_kg']):.2f}), largest change {G[k+'_dT'].max():.2f} K, multiplier {float(G[k+'_lam'])/100:.2f} euro/kg; "
          f"best constant point {float(G[k+'_constant_V']):.3f} V, {float(G[k+'_constant_T_in'])-273.15:.0f} °C: "
          f"{float(G[k+'_constant_cost']):.2f} cent ({100*(float(G[k+'_constant_cost'])/float(G[k+'_cost'])-1):.1f} % more)")

**What you should see.** The day's cost on the autumn weekday is 32.7263, 32.7259, 32.7265 and 32.7271 cent with the model of an hour on 100, 200, 400 and 800 cells: relative changes of 1.5e-05 and 1.9e-05, ten thousand times below the 0.5 % asked. They are too small to fall at a clean order (the upwind finite volumes converge irregularly in the current, as Ex10.2 found); what matters is that the cost is settled to 2e-05. The operating grid changes it by less than 6e-06 cent between 10 mV × 10 K and 2.5 mV × 2.5 K: the splines are exact far below the requirement. The finite volumes conserve steam and heat to rounding (7e-16 and 4e-15), and under a flat price the optimum is one operating point (zero spread), as section A4 says it must be.

On the four curves the optimal day costs 32.73, 33.69, 33.24 and 68.82 cent, makes exactly the contract's 93.33 g, and touches the 10 K limit (10.00 K) when checked by the finite volumes on 800 cells. The best operating point held all day (1.28 V at 750 °C on all four) costs 3.4 to 5.9 % more: that is what the hourly schedule is worth. The multiplier, the marginal cost of the contract's hydrogen, is 5.12, 5.59 and 5.90 euro/kg on the first three curves and 9.39 euro/kg in the cold snap.

In [ ]:
# The optimal day on the four price curves
fig, ax = plt.subplots(1, 3, figsize=(16, 3.6))
for name in mp.PRICE_NAMES:
    k = name.replace(" ", "_")
    ax[0].step(range(24), G[k + "_V"], where="mid", label=name)
    ax[1].step(range(24), G[k + "_T_in"] - 273.15, where="mid")
    ax[2].step(range(24), 1e3 * G[k + "_hydrogen_hour"], where="mid")
ax[0].set_ylabel("cell voltage [V]"); ax[1].set_ylabel("inlet temperature [°C]"); ax[2].set_ylabel("hydrogen [g/h]")
for a in ax:
    a.set_xlabel("hour of the day")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

**What you should see.** The voltage moves between the two edges of the band at 750 °C: **1.237 V**, where the cell cools by 10 K, in the dear hours, and **1.32 V**, where it heats by 10 K, in the cheap ones, with values between in the hours whose price is near the marginal one. The inlet temperature stays at its lower bound, 750 °C, almost all day: a cooler cell ages more slowly, and with the ageing counted its hydrogen is cheaper. Only in the cheapest night hours of the windy night and the cold snap is it raised, to 752 and 757 °C, to make more hydrogen at the top of the band. Each hour makes 3.36 to 4.62 g.

## C3 · The reference model: accuracy and time

The reference model is what an engineer would run with Ex10.2's solver: **SLSQP** (sequential quadratic programming, `scipy.optimize`) on the 48 numbers scaled to [0, 1], started from 1.25 V and 800 °C in every hour, the finite volumes on Ex10.2's **50 cells** as the model of every hour, the contract as an equality and the 24 limits as inequalities. Its derivatives are forward differences; because each hour depends only on its own two numbers, a gradient costs three solves an hour, 72 in all. It runs on the CPU, as the solver does. The cell below runs it on the four curves - a few minutes - and judges each schedule by the finite volumes on 800 cells.

In [ ]:
# The reference model on the four price curves, judged by the finite volumes on 800 cells
REF = {}
for name in mp.PRICE_NAMES:
    k = name.replace(" ", "_")
    r = mp.reference(mp.PRICES[name], demand=float(G["contract_kg"]))   # SLSQP with the finite volumes on 50 cells
    chk = mp.day(r["V"], r["T_in"], mp.PRICES[name])                     # judged on 800 cells
    REF[name] = dict(seconds=r["seconds"], iterations=r["iterations"], cost=chk["cost"], hydrogen=chk["hydrogen"], dT=chk["dT"].max())
    print(f"{name:>18}: {r['iterations']} SLSQP iterations, {r['seconds']:.1f} s; cost {chk['cost']:.2f} cent "
          f"({100*(chk['cost']/float(G[k+'_cost'])-1):+.2f} % against the ground truth), hydrogen "
          f"{100*(chk['hydrogen']/float(G['contract_kg'])-1):+.2f} % against the contract, largest change {chk['dT'].max():.2f} K")
t_ref = float(np.median([REF[n]["seconds"] for n in mp.PRICE_NAMES]))
print(f"\nthe reference model takes {t_ref:.1f} s per new price curve (median); the network must answer in under {t_ref/10:.1f} s")

**What you should see.** SLSQP converges in 9 to 13 iterations, 8 to 18 s per price curve on the authors' CPU (shared with other jobs): each iteration is 72 solves of the finite volumes. Judged on 800 cells its schedules cost 0.15 to 0.21 % more than the ground truth's, make 0.15 % more hydrogen than the contract (the 50-cell model underestimates the current by that much), and reach 10.11 K: **inside all three requirements**. Its error is that of its model of an hour, not of the optimiser. That is the bar: the network's schedule must be as good, ten times faster.

---

# Part D · The Physics-Informed Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **model of an hour** you optimise through (TODO 1), the **optimiser** that turns a price curve into a schedule (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else is given.

**The contract.**

| name | what it is |
|---|---|
| `model` | a PyTorch module used as `y, th, i = model(p)`, `p` an `(n, 3)` tensor of scaled $(x, v, \tau)$, like Ex10.2's network; `hour_numbers(model, V, T_in)` (section B1) turns it into the four numbers of each hour |
| `optimise_day(price)` | takes a NumPy array of 24 prices [euro/kWh] and returns two NumPy arrays of 24: the cell voltages [V] and the inlet temperatures [K] |

The evaluation calls `optimise_day` on each of the four price curves, times it, and judges the schedule it returns by the finite volumes on 800 cells. Whatever your optimiser needs - multipliers, penalties, margins, a warm start - stays inside it. It must not use the ground truth's schedules.

## D2 · Given: the contract and the objective

`H_CONTRACT` is the contract in grams; `day_numbers(model, V, T_in, price)` returns, for the 24 hours, the day's cost [cent], the hydrogen of the day [g] and the 24 temperature changes [K], all differentiable tensors.

In [ ]:
# Given: the contract, and the day's numbers through a model -----------------------
H_CONTRACT = 1e3 * float(G["contract_kg"])               # g of hydrogen a day

def day_numbers(model, V, T_in, price):
    # the day's cost [cent], hydrogen [g] and the 24 temperature changes [K] of a schedule, through the model
    h = hour_numbers(model, V, T_in)
    cost = (100 * price * h["electricity"] + h["ageing"]).sum()   # electricity at the hour's price, and ageing
    return cost, 1e3 * h["hydrogen"].sum(), h["dT"]

## D3 · TODO 1 · Your model of an hour

Make `model`: the network you optimise through. Ex10.2's trained network (`ex102`, section B1) is one choice; section B1 measured its errors in the four numbers of an hour. If you want it more exact where the schedule lives - in the band of section A3, near the 10 K limit - you may train it further, or train a network of your own with Ex10.2's loss; report the offline time. Then check it: the four numbers at a few operating points against the ground truth's table `G["table_..."]`.

In [ ]:
# TODO 1 - your model of an hour: model(p) -> (y, th, i), p = (x, v, tau) scaled to [0, 1]
model = ...
OFFLINE_SECONDS = ...                                    # the training time of your model [s]
for q in model.parameters():
    q.requires_grad_(False)                              # the model is finished: only the schedule moves now

## D4 · TODO 2 · Your optimiser

Write `optimise_day(price)`: 48 numbers, found by a gradient-based optimiser through `model`, its gradients by autograd (section B2 lists the choices: the bounds, the contract, the limit, the optimiser). Print what you need while you develop it, but leave it quiet for the evaluation, which runs it on the four curves several times to time it. `check_day(price, V, T_in)` below judges a schedule by the finite volumes on 800 cells, as the evaluation will.

In [ ]:
# A quick check, given: a schedule judged by the finite volumes on 800 cells against the ground truth
def check_day(name, V, T_in):
    k = name.replace(" ", "_")
    chk = mp.day(V, T_in, mp.PRICES[name])                # 24 solves on 800 cells, a second or two
    return (f"{name}: cost {100*(chk['cost']/float(G[k+'_cost'])-1):+.3f} % against the ground truth, hydrogen "
            f"{100*(chk['hydrogen']/float(G['contract_kg'])-1):+.3f} % against the contract, largest change {chk['dT'].max():.2f} K")

In [ ]:
# TODO 2 - your optimiser: price (24,) -> V (24,) [V], T_in (24,) [K]
def optimise_day(price):
    ...
    return V, T_in

V, T_in = optimise_day(mp.PRICES["autumn weekday"])
print(check_day("autumn weekday", V, T_in))

## D5 · TODO 3 · Your figures

Draw what you need to understand your optimiser and to answer the report: at least the course of the cost, the contract's shortfall and the largest temperature change during one optimisation, and one day's schedule against the ground truth's. Save each figure you want in the report as `MP10.2B_report_<name>.png` (in `cc.output_path`): part E embeds them in the PDF.

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell runs your `optimise_day` on each of the four price curves - each a new case - and keeps the answers in the dictionary `R`: the schedule, its **online time** (the middle of three runs, on this runtime, synchronised on a GPU), and the schedule judged by the finite volumes on 800 cells: the day's cost, the hydrogen and the largest temperature change. The reference model's times are those part C measured on the same runtime. Set `OFFLINE_SECONDS` to the training time of your model; it is reported beside the online times.

In [ ]:
# The evaluation, given ------------------------------------------------------------
out = {}
for name in mp.PRICE_NAMES:
    k = name.replace(" ", "_"); price = mp.PRICES[name]
    t = middle(lambda: optimise_day(price), reps=3)       # the online time of a new price curve
    V, T_in = optimise_day(price)
    chk = mp.day(V, T_in, price)                          # judged by the finite volumes on 800 cells
    out.update({f"{k}_V": np.asarray(V), f"{k}_T_in": np.asarray(T_in), f"{k}_seconds": np.array(t),
                f"{k}_cost": np.array(chk["cost"]), f"{k}_hydrogen": np.array(chk["hydrogen"]), f"{k}_dT": chk["dT"],
                f"{k}_ref_seconds": np.array(REF[name]["seconds"])})
    print(f"{name:>18}: network {t:.2f} s, reference {REF[name]['seconds']:.1f} s ({REF[name]['seconds']/t:.1f} times); "
          + check_day(name, V, T_in).split(": ", 1)[1])
out["offline_seconds"] = np.array(OFFLINE_SECONDS)
out["device"] = np.array(DEVICE)
R = out                                                  # part E reads the answers from here

---

# Part E · The Report

## E1 · What this part does

It scores your schedules, kept by section D6 in `R`, against the ground truth `G` and the reference model's times `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the twelve questions.

In [ ]:
# What part E uses, all from the parts above
print("in R:", ", ".join(R))
print("reference model (part C):", ", ".join(f"{n} {REF[n]['seconds']:.1f} s" for n in mp.PRICE_NAMES))

## E2 · TODO 4 · The goal metrics

Compute, against the ground truth, and put in the dictionary `METRICS` (a name and a number each; the report prints them as its results table):

* for each price curve: the error of the day's cost in per cent of the ground truth's, the hydrogen's error in per cent of the contract, the largest temperature change in kelvin, and your online time against the reference model's;
* what your schedule saves against the best constant point (`G["<curve>_constant_cost"]`), in per cent;
* the offline training time of your model.

Then say in one line each whether the four requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>60}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP10.2B_report_<name>.png` in `cc.output_path`, the figures your answers refer to. At least: your schedule against the ground truth's for the four curves (voltage and inlet temperature, with the price); the hydrogen made in every hour; the temperature change in every hour with the limit; and the online times of your optimiser and the reference model.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the twelve questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 7 are about your model, your optimiser and their result, 8 to 11 ask you to use them, 12 concludes. The next cell writes `MP10.2B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your model of an hour: which network, trained how and for how long, and its errors in the hydrogen, the electricity, the ageing and the temperature change against the ground truth's table, where the schedules live.": """
""",
    "2. Your optimiser: how the 48 numbers are parametrised, how the bounds, the contract and the 10 K limit are held, which optimiser, how many iterations, and why.": """
""",
    "3. The day's cost on the four price curves: your errors in per cent against the ground truth's optimum. Are they within 0.5 %? On which curve is your error largest, and why?": """
""",
    "4. The contract: the hydrogen your four schedules make, judged by the finite volumes, against the contract in per cent. Where does the difference come from?": """
""",
    "5. The limit: the largest temperature change of your four schedules, judged by the finite volumes. Which margin did you use, and what does a margin of zero give?": """
""",
    "6. The online time per new price curve: your optimiser against the reference model on the same runtime, in seconds, and the ratio. What does each spend its time on? How many price curves repay the offline training?": """
""",
    "7. What the schedule is worth: your saving against the best constant operating point on each curve, in per cent and in cent a day. Which curve gains most, and why?": """
""",
    "8. The shape of the schedule: in which hours does your cell run at the top of the voltage band, in which at the bottom, and where is the inlet temperature? Explain from the price, the ageing and the limit.": """
""",
    "9. The multiplier: the marginal cost of the contract's hydrogen on each curve, in euro/kg (from your optimiser or the ground truth). Compare it with Ex10.2's hydrogen price of 5 euro/kg: would a larger contract pay?": """
""",
    "10. Change the limit to 8 K and to 15 K in your optimiser (the network only): how much does the day's cost on the winter cold snap change? What does the limit cost the operator?": """
""",
    "11. Which of the numbers behind your schedule would you refuse to promise for a real stack, and which simplifications of the model (quasi-steady hours, ageing not fed back) would change the schedule most?": """
""",
    "12. To conclude: when would you schedule a day through the network rather than with SLSQP and the finite volumes, given your measured accuracy, online time and offline time? When would you not?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" if abs(v) >= 0.01 else f"| {k} | {v:.2e} |" for k, v in METRICS.items()]
    out = ["# MP10.2B — A Day of Operation Under an Hourly Price", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP10.2B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP10.2B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP10.2B_report.md into MP10.2B_report.pdf, with any figure
# saved as MP10.2B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP10.2B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP10.2B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP10.2B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP10.2B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass